# 01 — Data audit and exploratory analysis
This is a new executable rebuild, not a rearrangement of historical notebook outputs.

**Question:** Can available notification information distinguish records labelled `serious` from the other five known labels?

This is retrospective label classification. The source is a public notification snapshot, not all inspected products. Prediction-time availability of fields is not verified. All supplied product types are retained; Explorer can filter them but that does not retrain the model.

Start by installing `requirements.txt` in your Python environment. Open all four notebooks from this folder and execute 01 → 02 → 03 → 04. No API keys, paid services, LLM calls or model downloads are needed.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "rasff_core.py").exists():
    raise FileNotFoundError("Open this notebook from the extracted RASFF_Rebuild folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rasff_core import *


## 1. Use the historical clean snapshot, without hiding its provenance
The supplied `CleanedDataset.csv` retains original fields and pre-existing generated fields. The baseline uses only original fields. This workflow does not pretend that the clean snapshot is independently reconstructed from every raw row.

The raw CSV is inspected with `csv.reader` so malformed records are logged rather than silently skipped. Prior manual edits, including reference 2021.2621, are inherited and remain subject to source verification. Rows with invalid dates or unknown target labels are quarantined. Duplicate references cause an explicit error rather than an arbitrary choice.

In [2]:
df, audit = prepare_data()
display(pd.Series(audit).to_frame("Value"))
display(pd.read_csv(ARTIFACTS / "raw_malformed_rows.csv").head())

,Value
source,C:\Users\SAMSUNG\Downloads\RASFF_Rebuild_Execu...
source_sha256,108ee0009af0d194eb1609faf46b195eb9b2d7b3104a86...
loaded_rows,27397
retained_rows,27397
rejected_rows,0
raw_malformed_records,15
first_date,2020-01-02 09:24:48
last_date,2025-10-31 16:30:35
missing_hazards,7156
label_counts,"{'serious': 14756, 'not serious': 4134, 'undec..."


,ending_line,field_count,expected_fields,raw_values
0,6161,15,14,"[""2024.6416"", ""fruits and vegetables"", ""food"",..."
1,7131,16,14,"[""2024.4612"", ""food contact materials"", ""food ..."
2,7685,15,14,"[""2024.3652"", ""prepared dishes and snacks"", ""f..."
3,10356,16,14,"[""2023.7761"", ""soups, broths, sauces and condi..."
4,10427,15,14,"[""2023.7640"", ""meat and meat products (other t..."


## 2. Inspect missingness and label support
Missing `hazards` is not interpreted as safe. The subject may still describe a hazard. Missing strings are handled deterministically; corpus vocabularies and category frequencies are learned later, within training folds.

In [3]:
missing = df.select_dtypes(include=["string"]).eq("").sum().sort_values(ascending=False)
display(missing.to_frame("Empty values"))
display(df.risk_decision.value_counts().to_frame("Notifications"))
display(df.type.value_counts().to_frame("Product types"))

,Empty values
forFollowUp,13587
forAttention,12431
distribution,8638
hazards,7156
simplified_hazard,7156
origin,574
operator,94
subject,3
classification,0
risk_decision,0


,Notifications
risk_decision,
serious,14756
not serious,4134
undecided,3029
potential risk,2689
potentially serious,2372
no risk,417


,Product types
type,
food,24696
feed,1566
food contact material,1106
other,15
animals,10
"plants, plant products and other objects",4


## 3. Inventory the variables before selecting features
Usefulness and availability are different questions. Notification identifiers are metadata, not risk predictors. Distribution and cooperation fields may encode follow-up actions. Classification is linked to risk judgement and is reserved for an optional post-notification diagnostic scenario. Excluding it does not establish that every other public field was known before the decision.

`Hazard_Type` is an inherited LLM-generated field; `simplified_hazard` is an inherited simplified name. Neither is required. Optional comparisons use existing values only; they do not create or validate new LLM labels.

In [4]:
roles = {
 "subject": "Baseline text", "hazards": "Original text; missingness preserved",
 "category": "Baseline category", "type": "Baseline category",
 "origin": "Country tokens (multiple countries allowed)", "notifying_country": "Baseline category",
 "reference": "Identifier / audit only", "date": "Temporal splitting / dashboard",
 "year": "Dashboard; redundant with date", "month": "Dashboard; not a default predictor",
 "risk_decision": "Original target", "target": "Binary target: serious vs other labels",
 "classification": "Optional late diagnostic; excluded from primary model",
 "distribution": "Dashboard / follow-up; excluded from predictor",
 "forAttention": "Follow-up field; excluded", "forFollowUp": "Follow-up field; excluded",
 "operator": "Follow-up field; excluded", "Hazard_Type": "Optional derived-feature ablation",
 "simplified_hazard": "Optional derived-feature ablation",
}
display(pd.DataFrame([{"column":c,"role":roles.get(c,"Inherited derived field; not selected")} for c in df.columns]))

,column,role
0,reference,Identifier / audit only
1,category,Baseline category
2,type,Baseline category
3,subject,Baseline text
4,date,Temporal splitting / dashboard
5,notifying_country,Baseline category
6,classification,Optional late diagnostic; excluded from primar...
7,risk_decision,Original target
8,distribution,Dashboard / follow-up; excluded from predictor
9,forAttention,Follow-up field; excluded


## 4. Exploratory analysis describes notifications only
The entire snapshot can be displayed for descriptive exploration. Do not use patterns from the future test period to choose model settings. Notebook 02 confines model comparison to earlier dates. Counts are not prevalence or safety rates; no inspection-volume denominator is supplied.

In [5]:
import plotly.express as px
monthly = df.assign(period=df.date.dt.to_period("M").astype(str)).groupby(["period","risk_decision"]).size().reset_index(name="notifications")
px.bar(monthly,x="period",y="notifications",color="risk_decision",title="Recorded notifications over time").show()
display(pd.crosstab(df.category,df.risk_decision).head(15))

risk_decision,no risk,not serious,potential risk,potentially serious,serious,undecided
category,,,,,,
alcoholic beverages,4,9,17,4,41,14
animal by-products,0,7,9,2,3,1
bivalve molluscs and products thereof,2,15,16,103,355,50
cephalopods and products thereof,3,30,12,2,68,13
cereals and bakery products,50,270,195,200,855,209
"cocoa and cocoa preparations, coffee and tea",14,41,83,77,209,88
compound feeds,4,73,19,14,22,17
confectionery,4,104,115,73,301,36
crustaceans and products thereof,3,99,56,38,177,28


## 5. Lock non-overlapping time blocks
Split on whole calendar days so the same day is never in both sides of a boundary.
- First 60% of observed days: training and forward CV.
- Next 15%: validation diagnostic; final candidate selection rule is fixed beforehand using CV.
- Next 10%: sigmoid probability calibration only.
- Last 15%: final evaluation only.

These percentages refer to distinct dates, so row percentages differ. This is one reproducible design, not a universal optimal split.

In [6]:
blocks = temporal_blocks(df)
display(pd.DataFrame(json.loads((ARTIFACTS / "split_manifest.json").read_text())).T)
for a,b in zip(list(blocks)[:-1],list(blocks)[1:]):
    assert blocks[a].date.max() < blocks[b].date.min()
assert not set(blocks["train"].reference) & set(blocks["test"].reference)
print("Date and reference checks passed.")

,rows,first,last,serious_share
train,14668,2020-01-02 09:24:48,2023-05-17 18:41:05,0.591764
validation,4748,2023-05-18 10:57:14,2024-04-18 17:56:13,0.483783
calibration,3128,2024-04-19 08:14:07,2024-11-25 18:10:51,0.492008
test,4853,2024-11-26 09:45:15,2025-10-31 16:30:35,0.46157


Date and reference checks passed.


## Blog notes
Describe why the original target-encoding-before-split approach was replaced. Separate inherited data edits from new fixes. `data_audit.json` and `raw_malformed_rows.csv` are evidence, not proof that every original row was repaired correctly.